In [ ]:
!pip install datasets sentence-transformers bertopic


In [ ]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full", split="train")

In [ ]:
df = dataset.to_pandas()
df = df.sample(20000, random_state=42)
df.head()

In [ ]:
df = df.dropna(subset=["text"])
df["text"] = df["text"].str.lower()

In [ ]:
print("orginalshape:", dataset)
print("smaple shape:", df.shape)

In [ ]:
df.info()
df.isnull().sum()
df.dtypes

In [ ]:
df["label"].value_counts()

In [ ]:
from collections import Counter

words = " ".join(df["text"]).split()
Counter(words).most_common(20)

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text):
    words = text.split()
    words = [w for w in words if w not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
df["clean_text"]

In [ ]:
texts = df["clean_text"].tolist()

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic()

topics, probs = topic_model.fit_transform(texts, embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model = topic_model.reduce_topics(texts)

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(10)

In [ ]:
for topic in topic_info["Topic"][:8]:  # top 8 topics
    if topic == -1:
        continue

    print("\nTopic:", topic)
    print("Keywords:", topic_model.get_topic(topic))
    print("Sample Reviews:", topic_model.get_representative_docs(topic)[:2])

### Key Insights from Yelp Customer Reviews

1. Food & Restaurant Experience (Most Discussed Topic)
   - Customers frequently talk about food quality, taste, and overall dining experience.
   - This is the most dominant topic, indicating that restaurants are the primary focus of reviews.

2. Entertainment & Nightlife
   - Reviews include experiences related to clubs, music, dance shows, and nightlife activities.
   - Customers evaluate atmosphere, crowd, and overall enjoyment.

3. Beauty & Salon Services
   - Customers discuss haircuts, nail services, and salon experiences.
   - Feedback includes service quality, hygiene, and pricing.

4. Automotive Services
   - Reviews highlight issues with car repair, customer service, and communication.
   - Many complaints relate to delays and poor service handling.

5. Health & Wellness Services
   - Includes gyms, spas, massages, and fitness centers.
   - Customers focus on service quality and facilities.

6. Travel & Airport Experience
   - Reviews mention airport services, flights, and transportation.
   - Common issues include delays and service dissatisfaction.

7. Car Cleaning & Maintenance Services
   - Customers discuss car wash quality and cleanliness.
   - Complaints often relate to incomplete or poor cleaning.

In [ ]:
topic_info = topic_model.get_topic_info()

# Remove outlier topic (-1)
filtered = topic_info[topic_info["Topic"] != -1]

# Sort by count
filtered = filtered.sort_values(by="Count", ascending=False)

filtered.head()

Most Discussed Topic:
Food and restaurant-related reviews dominate the dataset,
indicating that dining experiences are the most frequently discussed by customers.

In [ ]:
filtered[["Topic", "Count"]]

In [ ]:
topic_model.visualize_barchart()

In [ ]:
topic_model.visualize_topics()

### Conclusion

This project demonstrates how unstructured customer reviews can be transformed into meaningful insights using NLP techniques such as embeddings and topic modeling.

The analysis reveals key customer concerns and dominant themes, helping businesses better understand customer feedback and improve services.